In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import os, gc
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning import LightningDataModule, LightningModule, Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, Timer

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader


from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import VotingRegressor

import warnings

warnings.filterwarnings("ignore")
pd.options.display.max_columns = None

In [ ]:
class CONFIG:
    seed = 42
    target_col = "responder_6"
    # feature_cols = ["symbol_id", "time_id"] + [f"feature_{idx:02d}" for idx in range(79)]+ [f"responder_{idx}_lag_1" for idx in range(9)]
    feature_cols = [f"feature_{idx:02d}" for idx in range(79)] + [
        f"responder_{idx}_lag_1" for idx in range(9)
    ]

In [ ]:
valid = pl.scan_parquet("validation.parquet").collect().to_pandas()
train = pl.scan_parquet("train.parquet").collect().to_pandas()

In [ ]:
X_train = train[CONFIG.feature_cols]
y_train = train[CONFIG.target_col]
w_train = train["weight"]
X_valid = valid[CONFIG.feature_cols]
y_valid = valid[CONFIG.target_col]
w_valid = valid["weight"]

In [ ]:
# Custom R2 metric for validation
def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (np.average((y_true) ** 2, weights=sample_weight) + 1e-38)
    return r2


class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, dropouts, lr, weight_decay, leak, batch_norm=True, activation='relu', lr_scheduler='cos_decay'):        super().__init__()
        self.save_hyperparameters()
        layers = []
        in_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(in_dim, hidden_dim))
            if batch_norm:
                layers.append(nn.BatchNorm1d(in_dim))
            if activation == 'relu':
                layers.append(nn.ReLU())
            elif activation == 'silu':
                layers.append(nn.SiLU())
            elif activation == 'gelu':
                layers.append(nn.GELU())
            elif activation == 'elu':
                layers.append(nn.ELU())
            elif activation == 'leaky_relu':
                layers.append(nn.LeakyReLU(leak))
            elif activation == 'tanh':
                layers.append(nn.Tanh())
            elif activation == 'sigmoid':
                layers.append(nn.Sigmoid())
            if i < len(dropouts):
                layers.append(nn.Dropout(dropouts[i]))
            layers.append(nn.Linear(in_dim, hidden_dim))
            # layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        layers.append(nn.Tanh())
        self.model = nn.Sequential(*layers)
        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []
        self.lr_scheduler = lr_scheduler

    def forward(self, x):
        return 5 * self.model(x).squeeze(-1)

    def training_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('train_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        return loss

    def validation_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('val_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        self.validation_step_outputs.append((y_hat, y, w))
        return loss

    def on_validation_epoch_end(self):
        """Calculate validation WRMSE at the end of the epoch."""
        y = torch.cat([x[1] for x in self.validation_step_outputs]).cpu().numpy()
        if self.trainer.sanity_checking:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
        else:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
            weights = torch.cat([x[2] for x in self.validation_step_outputs]).cpu().numpy()
            # r2_val
            val_r_square = r2_val(y, prob, weights)
            self.log("val_r_square", val_r_square, prog_bar=True, on_step=False, on_epoch=True)
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        if self.lr_scheduler == 'cos_decay':
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.trainer.max_epochs, eta_min=1e-6, last_epoch=-1)
        else:
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.316, patience=5,
                                                               verbose=True)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }

    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        epoch = self.trainer.current_epoch
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

def objective(trial):
    layers = trial.suggest_int("layers", 1, 6)
    hidden_dims = [trial.suggest_int("hidden_dim", 64, 256) for _ in range(layers)]
    dropouts = [trial.suggest_float(f"dropout_{i}", 0.1, 0.5) for i in range(layers)]
    batch_norm = trial.suggest_categorical("batch_norm", [True, False])
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    lr_scheduler = trial.suggest_categorical("lr_scheduler", ['cos_decay', 'plateau'])
    activation = trial.suggest_categorical("activation", ['relu', 'silu', 'gelu', 'elu', 'leaky_relu', 'tanh', 'sigmoid'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    leak = trial.suggest_float("leak", 0.01, 0.5)
    model = NN(len(CONFIG.feature_cols), hidden_dims, dropouts, lr, weight_decay, batch_norm, activation, leak, lr_scheduler)
    epochs = trial.suggest_int("epochs", 10, 500)
    trainer = Trainer(max_epochs=epochs, devices=1, accelerator='gpu', logger=False, callbacks=[EarlyStopping(monitor='val_loss', patience=4)])
    trainer.fit(model, train_dataloader, val_dataloader)
    return trainer.callback_metrics["val_r_square"].item()



In [ ]:
# train dataset
train_dataset = TensorDataset(
    torch.FloatTensor(X_train.values),
    torch.FloatTensor(y_train.values),
    torch.FloatTensor(w_train.values),
)
train_dataloader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
# validation dataset
val_dataset = TensorDataset(
    torch.FloatTensor(X_valid.values),
    torch.FloatTensor(y_valid.values),
    torch.FloatTensor(w_valid.values),
)
val_dataloader = DataLoader(val_dataset, batch_size=1024, shuffle=False)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

Do k-fold cv after finding params.